# PCD 노이즈 제거 → 메시 재건 파이프라인

스풀 PCD를 입력받아 **주변 노이즈를 제거하고 메시로 재건**하는 전 과정을 단계별로 시각화하며 검증합니다.

## 흐름

1. **로드** — PCD/PLY 불러오기 + 스케일 조정 (mm→m)
2. **통계적 이상치 제거** — 산발적으로 흩어진 노이즈 점 제거
3. **DBSCAN 클러스터링** — 멀리 떨어진 노이즈 덩어리(바닥·벽 등) 제거, 주 객체만 남김
4. **메시 재건** — Poisson / 마칭 큐브 두 방식으로 재건
5. **성능 확인** — 재건 메시와 원본 PCD 간 평균 거리 측정

각 단계는 open3d `draw_plotly`로 노트북 안에 인터랙티브 렌더링됩니다.

> 모든 핵심 함수는 [`util.pcd_tool`](../../python/util/pcd_tool.py)에 정의되어 있어, 검증 후 다른 코드에서 바로 import해 쓸 수 있습니다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1] / 'python'))  # util.pcd_tool 경로

import numpy as np
import open3d as o3d

# 같은 폴더의 파이프라인 모듈에서 함수 import
from util.pcd_tool import (
    load_pcd,
    remove_statistical_outliers,
    denoise_dbscan, cluster_dbscan, colorize_clusters,
    reconstruct_mesh_poisson, reconstruct_mesh_marching_cubes,
    mesh_pcd_distance,
    make_pcd, show3d, downsample_for_view, RGB,
)

np.random.seed(42)

## 1. PCD 로드 (스케일 조정)

`load_pcd(path, scale)` — 파일을 불러오고 단위를 변환합니다. 스풀 PCD가 mm 단위면 `SCALE = 1e-3`으로 m로 맞춰야 이후 거리 기반 파라미터(eps 등)가 의미를 가집니다.

In [ ]:
# ====== 설정 ======
PCD_PATH = "../../sample/MERGED_SPOOL-004_0001_20260528T170812.pcd"
SCALE    = 1e-3   # mm→m
# ==================

pcd_raw = load_pcd(PCD_PATH, scale=SCALE)
pts = np.asarray(pcd_raw.points)
print(f"로드: {len(pts):,} 점")
print(f"범위: {pts.min(0).round(3)} ~ {pts.max(0).round(3)} m")

show3d([make_pcd(downsample_for_view(pcd_raw), 'gray')])
print('1. 원본 PCD')

## 2. 통계적 이상치 제거

`remove_statistical_outliers(pcd, nb_neighbors, std_ratio)` — 각 점의 이웃 평균 거리가 전체 분포에서 `std_ratio`σ 이상 벗어나면 노이즈로 보고 제거합니다. 측정 과정에서 생긴 **산발적인 떠다니는 점**을 걸러냅니다.

- `nb_neighbors`: 거리 계산에 쓸 이웃 수
- `std_ratio`: 작을수록 공격적으로 제거

In [ ]:
pcd_stat, kept_idx = remove_statistical_outliers(pcd_raw, nb_neighbors=20, std_ratio=2.0)
n_removed = len(pcd_raw.points) - len(pcd_stat.points)
print(f"제거된 산발 노이즈: {n_removed:,} 점 → 남은 점 {len(pcd_stat.points):,}")

# 제거된 점(빨강) vs 유지된 점(회색)
removed = pcd_raw.select_by_index(kept_idx, invert=True)
show3d([
    make_pcd(downsample_for_view(pcd_stat), 'gray'),
    make_pcd(removed, 'red'),
])
print('2. 통계적 이상치 제거 (빨강 = 제거된 점)')

## 3. DBSCAN 클러스터링으로 주변 노이즈 제거

`denoise_dbscan(pcd, eps, min_points, keep)` — 점들을 밀도 기반으로 클러스터링한 뒤, 주 객체(가장 큰 클러스터)만 남기고 바닥·벽·인접 구조물 같은 **떨어진 덩어리 노이즈**를 제거합니다.

- `eps`: 같은 클러스터로 묶을 최대 점간 거리 (m). **점 밀도에 맞게 조정**이 가장 중요
- `min_points`: 코어 점 최소 이웃 수
- `keep`: `'largest'`(최대 클러스터만) 또는 `'all_valid'`(`min_cluster_size` 이상 모두)

먼저 클러스터를 색으로 확인한 뒤 필터링합니다.

In [ ]:
EPS        = 0.05   # 점 밀도에 맞게 조정 (너무 작으면 과분할, 크면 노이즈까지 흡수)
MIN_POINTS = 15

# 3-1. 클러스터 라벨을 색으로 시각화
labels = cluster_dbscan(pcd_stat, eps=EPS, min_points=MIN_POINTS)
n_clusters = labels.max() + 1
print(f"클러스터 수: {n_clusters}  (노이즈로 분류된 점: {(labels < 0).sum():,})")

# 전체를 색칠한 뒤 다운샘플 (색-점 대응 유지)
colored = colorize_clusters(pcd_stat, labels)
show3d([downsample_for_view(colored)])
print('3-1. DBSCAN 클러스터 (색깔별 = 클러스터, 검정 = 노이즈)')

In [ ]:
# 3-2. 가장 큰 클러스터만 남기기
pcd_clean, _ = denoise_dbscan(pcd_stat, eps=EPS, min_points=MIN_POINTS, keep='largest')
print(f"최종 정제: {len(pcd_clean.points):,} 점")

show3d([make_pcd(downsample_for_view(pcd_clean), 'blue')])
print('3-2. 노이즈 제거 완료 (주 객체만)')

## 4. 메시 재건

정제된 PCD로 표면 메시를 만듭니다. 두 방식을 비교합니다.

### 4-1. Poisson 표면 재건
`reconstruct_mesh_poisson(pcd, depth)` — 법선을 추정한 뒤 Poisson 방정식으로 매끄러운 폐곡면을 생성하고, 데이터가 없는 저밀도 영역은 잘라냅니다. `depth`가 클수록 디테일↑·연산량↑.

In [ ]:
mesh_poisson = reconstruct_mesh_poisson(pcd_clean, depth=9)
print(f"Poisson 메시: 정점 {len(mesh_poisson.vertices):,}  면 {len(mesh_poisson.triangles):,}")

mesh_poisson.paint_uniform_color(RGB['green'])
show3d([mesh_poisson])
print('4-1. Poisson 재건 메시')

### 4-2. 마칭 큐브 재건
`reconstruct_mesh_marching_cubes(pcd, resolution, sigma, level)` — 복셀 그리드 + 가우시안 스무딩 후 등가면을 추출합니다 (`pcd_ply_to_mesh.points_to_mesh` 재사용). `resolution`↑ = 디테일↑·메모리↑.

In [ ]:
mesh_mc = reconstruct_mesh_marching_cubes(pcd_clean, resolution=128, sigma=1.5, level=0.5)
print(f"마칭큐브 메시: 정점 {len(mesh_mc.vertices):,}  면 {len(mesh_mc.triangles):,}")

mesh_mc.paint_uniform_color(RGB['orange'])
show3d([mesh_mc])
print('4-2. 마칭 큐브 재건 메시')

## 5. 재건 성능 확인

`mesh_pcd_distance(mesh, pcd)` — 재건 메시 표면을 샘플링해, 원본(정제) PCD 각 점에서 **가장 가까운 메시 점까지의 거리**를 측정합니다. 평균/RMS가 작을수록 원본을 잘 재현한 것입니다.

In [ ]:
d_poisson = mesh_pcd_distance(mesh_poisson, pcd_clean)
d_mc      = mesh_pcd_distance(mesh_mc, pcd_clean)

print(f"{'방식':<12}{'mean (m)':>12}{'rms (m)':>12}{'max (m)':>12}{'면 수':>12}")
print('-' * 60)
print(f"{'Poisson':<12}{d_poisson['mean']:>12.5f}{d_poisson['rms']:>12.5f}"
      f"{d_poisson['max']:>12.5f}{len(mesh_poisson.triangles):>12,}")
print(f"{'MarchingCube':<12}{d_mc['mean']:>12.5f}{d_mc['rms']:>12.5f}"
      f"{d_mc['max']:>12.5f}{len(mesh_mc.triangles):>12,}")

In [ ]:
# 원본 점(회색) + 재건 메시(반투명 초록) 겹쳐 보기
show3d([
    make_pcd(downsample_for_view(pcd_clean), 'gray'),
    mesh_poisson,
])
print('5. 원본 점(회색) vs Poisson 메시 — 겹침 정도 확인')

## 요약 & 함수 재사용

검증한 파이프라인 전체를 [`util.pcd_tool`](../../python/util/pcd_tool.py)에서 import해 한 번에 쓸 수 있습니다.

```python
from util.pcd_tool import (
    load_pcd, remove_statistical_outliers, denoise_dbscan,
    reconstruct_mesh_poisson, mesh_pcd_distance,
)

pcd  = load_pcd("spool.pcd", scale=1e-3)
pcd, _ = remove_statistical_outliers(pcd)
pcd, _ = denoise_dbscan(pcd, eps=0.05, min_points=15, keep='largest')
mesh   = reconstruct_mesh_poisson(pcd, depth=9)
print(mesh_pcd_distance(mesh, pcd))
```

| 단계 | 함수 | 핵심 파라미터 |
|------|------|----------------|
| 로드 | `load_pcd` | `scale` |
| 산발 노이즈 제거 | `remove_statistical_outliers` | `nb_neighbors`, `std_ratio` |
| 덩어리 노이즈 제거 | `denoise_dbscan` | `eps`, `min_points`, `keep` |
| 메시 재건 | `reconstruct_mesh_poisson` / `reconstruct_mesh_marching_cubes` | `depth` / `resolution`,`sigma`,`level` |
| 성능 평가 | `mesh_pcd_distance` | — |